<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_6/lessons/lesson_53_graduation_pitch/note_lesson_53_pitch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Урок 53 — Питч фінального проєкту

Випускна презентація — це **питч**: ти продаєш ідею інвестору, клієнту чи техліду, що наймає. Більшість рішень про питч можна перевірити кодом, як ми перевіряли архітектуру в уроці 52: дерево «чому», питання для інтерв'ю, розрахунок ринку, дек у Markdown, час виступу, критерії оцінювання. Скрипт ловить **форму**; чи переконливий зміст — вирішують люди. Приклад наскрізний: питч «Тема дня» на основі `news_hub` (Олена — вигаданий персонаж).

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | підготовка: `pitch_check.py` з папки уроку | 0.1 | |
| 1 | кому ти продаєш: акценти й прохання | 1.1 | |
| 2 | **5 «чому»** як дерево: корені, ланцюжок, зворотне читання, технічна проблема | 2.1–2.4 | |
| 3 | **The Mom Test**: погані й кращі питання, факти чи компліменти | 3.1–3.2 | **Задача 1** |
| 4 | **ринок**: згори вниз проти знизу вгору, чутливість, економіка одного користувача | 4.1–4.3 | **Задача 2** |
| 5 | **дек як дані**: слайди, слова на слайді, `check()`, CLI, план часу, «зміни приклад», дек із шаблону | 5.1–5.7 | **Задача 3** |
| 6 | знайди помилку: чернетка деку і слайд «Ринок» | 6.1–6.2 | |
| 7 | оцінювання і репетиції як дані; Marp | 7.1–7.3 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, константи, тексти слайдів) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Повний розбір — структури деків Sequoia, Y Combinator і Кавасакі, 5 «чому» з Toyota та його межі, демо, питання після виступу, оцінювання: [Урок 53](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_53/).

---
## 0. Підготовка

### Приклад 0.1. Скрипт перевірки деку

`pitch_check.py` лежить у папці уроку. `find_course_root` шукає репозиторій курсу вгору від поточної папки, а в Colab клонує лише папку уроку. Тимчасові файли (свій дек із шаблону) — у `WORK`, прибираємо наприкінці.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
LESSON = "module_6/lessons/lesson_53_graduation_pitch"


def find_course_root(marker: str, sparse: list[str]) -> Path:
    """Корінь репозиторію курсу: шукаємо вгору від поточної папки; не знайшли (Colab) — клонуємо лише потрібні папки."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / marker).is_dir():
            return folder
    target = here / "course_repo"
    if not target.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, str(target)],
                       check=True)
        subprocess.run(["git", "-C", str(target), "sparse-checkout", "set", *sparse], check=True)
    return target

In [ ]:
ROOT = find_course_root(LESSON, [LESSON])
HERE = ROOT / LESSON
sys.path.insert(0, str(HERE))
from pitch_check import MAX_WORDS_ON_SLIDE, REQUIRED, WORDS_PER_MINUTE, check, split_slides, words

EXAMPLE = (HERE / "example_news_digest_deck.md").read_text(encoding="utf-8")
TEMPLATE = (HERE / "pitch_deck_template.md").read_text(encoding="utf-8")
WORK = Path(tempfile.mkdtemp(prefix="pitch_"))
print("темп мови:", WORDS_PER_MINUTE, "слів/хв | ліміт на слайді:", MAX_WORDS_ON_SLIDE, "слів")
print("обов'язкові розділи:", REQUIRED)

---
## 1. Кому ти продаєш

Та сама ідея продається різним людям по-різному. Слайди майже ті самі — змінюються **акценти** і **прохання** наприкінці.

### Приклад 1.1. Таблиця слухачів як дані

Задача: перед виступом швидко відповісти «на що тиснути і що просити». Таблиця з книги стає словником, а відповідь — функцією.

In [ ]:
AUDIENCES = {
    "інвестор": {"питання": "чи виросте це у великий бізнес?",
                 "акцент": ["ринок", "чому зараз", "команда", "тяга"], "прохання": "сума і на що підуть гроші"},
    "клієнт": {"питання": "чи розв'яже це мою проблему дешевше, ніж зараз?",
               "акцент": ["проблема", "демо", "ціна", "ризики впровадження"], "прохання": "пілот, перший контракт"},
    "техлід": {"питання": "чи вміє ця людина робити рішення і перевіряти їх?",
               "акцент": ["архітектура", "компроміси", "тести", "виміри"], "прохання": "співбесіда, код-рев'ю"},
}


def pitch_focus(audience: str) -> str:
    """Підказка перед виступом: головне питання слухача, акценти і прохання."""
    if audience not in AUDIENCES:
        return "не знаю, хто в залі → базовий дек: проблема → демо → як працює → одне прохання"
    a = AUDIENCES[audience]
    return f"{audience}: «{a['питання']}» → акцент: {', '.join(a['акцент'])} → прохання: {a['прохання']}"

In [ ]:
for who in ("інвестор", "клієнт", "техлід", "викладач і група"):
    print(pitch_focus(who))

На уроці 53 у залі всі троє одночасно: викладач, група і, можливо, гості. Тому базовий дек з 10 слайдів покриває всі три питання, а акцент ти обираєш сам.

---
## 2. 5 «чому» як дерево

Тайічі Оно, творець системи виробництва Toyota, радив питати «чому» п'ять разів. Пастка методу — думати, що причина одна. На кожне «чому» буває кілька відповідей, і це **дерево**: вузол — твердження, діти — відповіді на «чому?». **Корені** — листки дерева: причини у світі, які можна змінити. «Немає мого продукту» — не корінь: це рішення, переодягнене в причину.

### Приклад 2.1. Дерево Олени

Задача: побачити дерево цілком, а не один ланцюжок. Словник `{твердження: [відповіді на «чому?»]}` і рекурсивний друк з відступами.

In [ ]:
WHY = {
    "пропускає важливі новини": ["переглядає новини раз на день", "сповіщення каналів тонуть у чатах", "«неуважна»"],
    "переглядає новини раз на день": ["її теми розкидані по різних рубриках"],
    "її теми розкидані по різних рубриках": ["новини впорядковані за рубриками редакції, а не за темами читача"],
    "сповіщення каналів тонуть у чатах": ["кожен канал сповіщає однаково гучно"],
}
LABELS = {"«неуважна»"}                                   # ярлик, а не причина: змінити нічого


def show_tree(tree: dict[str, list[str]], node: str, labels: set[str], depth: int = 0) -> None:
    """Дерево «чому» з відступами; листки позначені як корінь або ярлик."""
    children = tree.get(node, [])
    mark = "" if children else ("   ← ярлик, не корінь" if node in labels else "   ← корінь")
    print("    " * depth + ("чому? → " if depth else "") + node + mark)
    for child in children:
        show_tree(tree, child, labels, depth + 1)

In [ ]:
show_tree(WHY, "пропускає важливі новини", LABELS)

Три гілки — три різні історії. Дві закінчуються причинами, які можна змінити, одна — ярликом.

### Приклад 2.2. Корені і ланцюжок до кореня

Задача: зібрати корені автоматично і показати шлях від симптому до кореня — саме цей шлях ти розповіси на слайді «Проблема».

- `roots(tree, node, labels)` — листки, досяжні з `node` (у порядку обходу), без ярликів;
- `path(tree, start, goal)` — список вузлів від `start` до `goal` або `[]`, якщо дороги немає.

In [ ]:
def roots(tree: dict[str, list[str]], node: str, labels: set[str]) -> list[str]:
    """Листки дерева, досяжні з node, крім ярликів."""
    children = tree.get(node, [])
    if not children:
        return [] if node in labels else [node]
    return [leaf for child in children for leaf in roots(tree, child, labels)]


def path(tree: dict[str, list[str]], start: str, goal: str) -> list[str]:
    """Ланцюжок «чому» від start до goal (обхід у глибину) або []."""
    if start == goal:
        return [start]
    for child in tree.get(start, []):
        rest = path(tree, child, goal)
        if rest:
            return [start] + rest
    return []

In [ ]:
found = roots(WHY, "пропускає важливі новини", LABELS)
print("корені:", found)
chain = path(WHY, "пропускає важливі новини", found[0])
print(" → чому? → ".join(chain), f"({len(chain) - 1} «чому»)")
assert found == ["новини впорядковані за рубриками редакції, а не за темами читача", "кожен канал сповіщає однаково гучно"]
assert len(chain) == 4 and path(WHY, "«неуважна»", found[0]) == []

🔮 **Прогноз:** ланцюжок до першого кореня — лише 3 «чому», а не 5. Це помилка?

<details>
<summary>Відповідь</summary>

Ні. «П'ять» — спостереження Оно, скільки разів зазвичай вистачає, а не правило. Зупиняються тоді, коли відповідь — це те, що можна змінити. Два корені — два різні продукти: для питчу обирають один.
</details>

### Приклад 2.3. Зворотне читання: «тому»

Задача: перевірити ланцюжок. Невелика зміна — читаємо його від кореня до симптому, вставляючи «тому». Якщо кожен крок звучить правдоподібно, ланцюжок тримається; якщо десь «тому» звучить дивно — там дірка в логіці.

In [ ]:
def read_back(chain: list[str]) -> str:
    """Ланцюжок від кореня до симптому через «тому»."""
    steps = chain[::-1]
    return steps[0] + "".join(f"\n   → тому {step}" for step in steps[1:])

In [ ]:
print(read_back(chain))
print()
print(read_back(path(WHY, "пропускає важливі новини", found[1])))

«Новини впорядковані за рубриками → тому теми розкидані → тому переглядає раз на день → тому пропускає» — звучить як історія. Саме її Олена має **підтвердити** в розмові (розділ 3): відповідь на «чому» треба побачити, а не вигадати за столом (*genchi genbutsu* — «піди й подивись сам»).

### Приклад 2.4. Технічна проблема: симптом чи корінь

Той самий метод працює для помилок у коді. Урок 52 знайшов: з двома репліками вебсервера ліміт входу пропускав 10 спроб замість 5. Задача: порівняти два виправлення **числом**, а не думкою. Модель: якщо кеш у пам'яті процесу, кожна репліка рахує свій ліміт.

In [ ]:
TECH_WHYS = [
    ("симптом", "зловмисник має 10 спроб входу замість 5"),
    ("чому 10?", "кожна з двох реплік рахує свої 5 спроб"),
    ("чому свої?", "лічильник throttle DRF лежить у кеші Django"),
    ("чому кеш не спільний?", "типовий кеш Django — пам'ять процесу (LocMemCache)"),
    ("чому ніхто не помітив?", "тести запускають один процес — там усе правильно"),
    ("чому немає перевірки? · корінь", "стек з кількома процесами ніхто не перевіряв на ліміти"),
]


def allowed_attempts(limit: int, replicas: int, shared_cache: bool) -> int:
    """Скільки спроб входу отримає зловмисник: спільний кеш — limit, свій у кожної репліки — limit × replicas."""
    return limit if shared_cache else limit * replicas

In [ ]:
for question, answer in TECH_WHYS:
    print(f"{question:<32} {answer}")
print()
for fix, limit, shared in (("як є", 5, False), ("симптом: ліміт 3", 3, False), ("корінь: кеш у Redis", 5, True)):
    print(f"{fix:<20} 2 репліки → {allowed_attempts(limit, 2, shared)} спроб, 4 репліки → {allowed_attempts(limit, 4, shared)}")

«Ліміт 3» лише ховає проблему: з чотирма репліками знову 12 спроб, а чесні користувачі тепер блокуються після трьох помилок. Виправлення кореня (спільний кеш + перевірка стеку з `--scale web=2`) дає рівно 5 за будь-якої кількості реплік. Для технічного слайда це сильна історія: проблема, корінь, вимір.

---
## 3. Перевірити проблему: The Mom Test

5 «чому» дають **гіпотезу**. Перевіряють її розмовою з людьми, які мають цю проблему. Пастка: люди ввічливі — спитай маму, чи гарна твоя ідея, і вона скаже «так». Три правила Роба Фіцпатріка: говори про **їхнє життя**, а не свою ідею; питай про **конкретне минуле**, а не майбутнє; менше говори.

### Приклад 3.1. Погане питання → краще

In [ ]:
REWRITES = [
    ("Вам сподобався б бот із новинами?", "майбутнє + твоя ідея", "Як ви дізналися про останню важливу новину своєї галузі?"),
    ("Ви б платили 99 грн на місяць?", "обіцянка без зобов'язань", "За які сервіси для роботи ви вже платите? Скільки?"),
    ("Це ж проблема — пропускати новини?", "навідне", "Коли востаннє ви дізналися про новину запізно? Що сталося потім?"),
    ("Скільки часу ви б зекономили?", "фантазія", "Скільки часу вчора пішло на перегляд новин?"),
]

In [ ]:
for bad, why, better in REWRITES:
    print(f"✗ {bad:<38} ({why})\n  ✓ {better}")

Усі кращі питання — про те, що людина **вже робила**: вчора, востаннє, за що вже платить.

### Задача 1. Детектор поганих питань

Напиши `problems(question)` → множину причин, чому питання погане (порівнюй без урахування регістру):

- `"майбутнє"` — є частка «б» / «би» **окремим словом** (умовний спосіб: «користувались би», «ви б платили»);
- `"твоя ідея"` — згадано продукт: одне зі слів `IDEA_WORDS` окремим словом;
- `"навідне"` — питання починається з «це ж» або містить «правда ж».

Добре питання повертає порожню множину. Слова зручно діставати так само, як `pitch_check.words`: `re.findall(r"[\w'’]+", text.lower())`.

In [ ]:
IDEA_WORDS = {"бот", "ботом", "бота", "додаток", "наш", "мій", "продукт"}
QUESTIONS = {
    "Ви б користувались ботом з новинами?": {"майбутнє", "твоя ідея"},
    "Це ж проблема — пропускати новини?": {"навідне"},
    "Скільки ви платили б за наш додаток?": {"майбутнє", "твоя ідея"},
    "Як ви дізналися про останню важливу новину своєї галузі?": set(),
    "Скільки часу вчора пішло на перегляд новин?": set(),
    "За які сервіси для роботи ви вже платите?": set(),
    "Мій продукт вам допоміг би, правда ж?": {"майбутнє", "твоя ідея", "навідне"},
    "Хто вибирав, де ви берете новини, — ви чи бібліотека?": set(),    # «би» всередині слова — не частка
}

In [ ]:
def problems(question: str) -> set[str]:
    # BEGIN SOLUTION
    text = question.lower()
    found_words = set(re.findall(r"[\w'’]+", text))
    found = set()
    if {"б", "би"} & found_words:
        found.add("майбутнє")
    if IDEA_WORDS & found_words:
        found.add("твоя ідея")
    if text.startswith("це ж") or "правда ж" in text:
        found.add("навідне")
    return found
    # END SOLUTION

In [ ]:
for question, expected in QUESTIONS.items():
    got = problems(question)
    print(("✓" if got == expected else "✗"), f"{question:<58}", sorted(got) if got else "добре")
assert isinstance(problems("Як справи?"), set), "поверни множину причин"
for question, expected in QUESTIONS.items():
    assert problems(question) == expected, f"{question!r}: очікувалось {sorted(expected)}, отримано {sorted(problems(question))}"
print("✅ Задача 1 розв'язана")

Правила-слова — лише перший фільтр (питання «Що ви робили б?» він зловить, а «Уявіть, що є сервіс…» — ні). Але навіть такий фільтр корисний перед інтерв'ю: прогнати свої 5 питань і переписати ті, що «світяться».

### Приклад 3.2. Відповідь: факт чи комплімент

Задача: після розмови розібрати нотатки. Справжня перевірка — **відповідь**: якщо людина розповідає, що вона робила («вчора відкрила шість сайтів»), — це дані. Якщо каже, що **зробила б** («цікаво, користувалась би»), — це компліменти. Невелика зміна до задачі 1: дивимось не на питання, а на відповідь — умовний спосіб і слова-оцінки проти чисел і конкретного часу.

In [ ]:
COMPLIMENT = {"б", "би", "цікаво", "класно", "круто", "звучить", "прикольно"}
PAST_MARKERS = {"вчора", "минулого", "востаннє", "щоранку", "позавчора", "тиждень"}


def answer_kind(answer: str) -> str:
    """'комплімент', 'факт' або 'уточнити' — за словами відповіді."""
    found_words = set(re.findall(r"[\w'’]+", answer.lower()))
    if found_words & COMPLIMENT:
        return "комплімент"
    if found_words & PAST_MARKERS or re.search(r"\d", answer):
        return "факт"
    return "уточнити"


ANSWERS = [
    "Так, цікаво, я б користувалась.",
    "Вчора відкрила шість сайтів, а новину про тариф побачила аж в обід.",
    "Плачу 300 грн на місяць за моніторинг згадок, але він ловить лише назву компанії.",
    "Ну, новини — це важливо.",
]

In [ ]:
for answer in ANSWERS:
    print(f"{answer_kind(answer):<11} {answer}")

Найсильніший доказ — третя відповідь: людина **вже платить** за розв'язання цієї проблеми. На слайді «Проблема» один такий факт важить більше, ніж десять «цікава ідея». «Уточнити» — сигнал перепитати: «А коли востаннє це було важливо? Що ви тоді зробили?»

---
## 4. Ринок знизу вгору

- **TAM** — усі, хто має проблему;
- **SAM** — кого може обслужити продукт (мова, країна, канал);
- **SOM** — кого досягнеш за рік.

Дохід = SOM × ціна × 12. Кожне число — або з джерелом, або **чесно позначене припущення**.

### Приклад 4.1. Згори вниз: велике число без жодної перевірки

Задача: побачити, чому слайд «ринок — мільярди, візьмемо 1%» слабкий. Невелика зміна — порахуємо його так само чесно, як порахуємо знизу вгору: скільки входів має джерело.

In [ ]:
def top_down(market_usd: float, share: float) -> str:
    """Розрахунок «частка від світового ринку» і скільки в ньому перевірюваних частин."""
    revenue = market_usd * share
    checkable = 0                                   # ні «мільярдів», ні «1%» перевірити нічим
    return f"{market_usd / 1e9:.0f} млрд $ × {share:.0%} = {revenue / 1e6:.0f} млн $ на рік; перевірюваних входів: {checkable}"

In [ ]:
print(top_down(5e9, 0.01))

50 мільйонів доларів «з повітря»: невідомо, хто ці люди, чому прийдуть саме вони і звідки «мільярди». Інвестор спитає — і довіра до решти слайдів впаде.

### Задача 2. Ринок знизу вгору з позначеними припущеннями

Напиши `bottom_up(inputs)`. Кожен вхід — пара `(значення, джерело)`, де джерело `None` означає «припущення». Входи йдуть у порядку: людей з проблемою, частка в Telegram, частка SAM за рік, ціна за місяць. Поверни словник з ключами `TAM`, `SAM`, `SOM` (цілі, округлені вниз: `math.floor`), `дохід_рік` (`SOM × ціна × 12`) і `припущення` — **відсортований** список назв входів без джерела.

In [ ]:
import math

INPUTS = {
    "фахівців PR, маркетингу, аналітики": (40_000, None),
    "частка, що читає новини в Telegram": (0.72, "Internews, 2025"),
    "частка SAM, яку досягнемо за рік": (0.01, None),
    "ціна за місяць, грн": (99, None),
}

In [ ]:
def bottom_up(inputs: dict[str, tuple[float, str | None]]) -> dict[str, object]:
    # BEGIN SOLUTION
    people, telegram, reach, price = (value for value, _ in inputs.values())
    tam = math.floor(people)
    sam = math.floor(tam * telegram)
    som = math.floor(sam * reach)
    return {"TAM": tam, "SAM": sam, "SOM": som, "дохід_рік": som * price * 12,
            "припущення": sorted(name for name, (_, source) in inputs.items() if source is None)}
    # END SOLUTION

In [ ]:
market = bottom_up(INPUTS)
assert isinstance(market, dict) and {"TAM", "SAM", "SOM", "дохід_рік", "припущення"} <= set(market), "поверни словник з усіма п'ятьма ключами"
for key, value in market.items():
    print(f"{key:<12} {value}")
assert (market["TAM"], market["SAM"], market["SOM"]) == (40_000, 28_800, 288), "TAM → SAM = TAM × частка Telegram → SOM = SAM × частка за рік"
assert market["дохід_рік"] == 288 * 99 * 12 == 342_144
assert market["припущення"] == sorted(market["припущення"]) and len(market["припущення"]) == 3
print("✅ Задача 2 розв'язана")

🔮 **Прогноз:** три з чотирьох чисел — припущення. Чи варто тоді взагалі ставити цей слайд?

<details>
<summary>Відповідь</summary>

Так — але з позначками. Інвестор не чекає точності від проєкту без користувачів; він перевіряє, чи ти **вмієш рахувати** і чи **відрізняєш** знання від здогадок. 40 000 фахівців — число, яке ти перевіриш першим; 1% — найсумнівніше, і саме його підтвердить або спростує пілот.
</details>

### Приклад 4.2. Чутливість: які припущення важать найбільше

Задача: на питання «а якщо не 1%?» відповісти таблицею, а не «ну, приблизно». Невелика зміна — та сама формула, але перебираємо найсумнівніші входи: частку за рік і ціну.

In [ ]:
def yearly_revenue(people: int, telegram: float, reach: float, price: int) -> int:
    som = math.floor(math.floor(people * telegram) * reach)
    return som * price * 12


REACH = (0.005, 0.01, 0.02)
PRICES = (49, 99, 149)

In [ ]:
print("частка за рік │ " + " │ ".join(f"{p:>3} грн/міс" for p in PRICES))
for reach in REACH:
    row = " │ ".join(f"{yearly_revenue(40_000, 0.72, reach, price):>11,}".replace(",", " ") for price in PRICES)
    print(f"{reach:>12.1%} │ {row}")

Дохід змінюється у **дванадцять разів** між кутами таблиці (від ~85 тис. до ~1 млн грн на рік). Подвоїти частку — те саме, що подвоїти ціну, але частку ти знаєш найгірше. Тому пілот, що перевіряє, скільки людей справді залишаються, важливіший за дискусію про ціну.

### Приклад 4.3. Скільки коштує один користувач

Питання після виступу №4: «Скільки коштує один користувач (сервер, виклики LLM)?». Задача: мати відповідь числом. Усі числа нижче — **припущення**, і так їх і треба назвати вголос.

In [ ]:
COSTS = {                                       # усе — припущення для прикладу
    "сервер, грн/міс": 600,                      # одна VPS для api + PostgreSQL + Redis
    "дайджестів на користувача за місяць": 30,
    "ціна одного виклику LLM, грн": 0.4,
}


def unit_economics(users: int, price: int, costs: dict[str, float]) -> dict[str, float]:
    """Витрати й маржа на одного користувача за місяць."""
    server = costs["сервер, грн/міс"] / users
    llm = costs["дайджестів на користувача за місяць"] * costs["ціна одного виклику LLM, грн"]
    cost = server + llm
    return {"сервер": round(server, 2), "LLM": round(llm, 2), "разом": round(cost, 2),
            "маржа": round(price - cost, 2), "маржа, %": round((price - cost) / price * 100)}

In [ ]:
for users in (50, 288, 2000):
    print(f"користувачів {users:>5}: {unit_economics(users, 99, COSTS)}")

На 50 користувачах сервер — половина витрат (12 з 24 грн), на 288 (наш SOM) маржа вже 86%, а далі головна стаття — виклики LLM (12 грн), і вона з ростом не зменшується. Звідси два технічні рішення, які можна показати на слайді «Як це працює»: кеш аналізу за хешем тексту (урок 44) і один дайджест на тему, а не на користувача.

---
## 5. Дек як дані

Дек у Markdown для Marp — це текст: `---` розділяє слайди, `<!-- … -->` — те, що кажеш уголос (нотатки доповідача). Отже, його можна розібрати й перевірити кодом — як `pitch_check.py` з папки уроку.

### Приклад 5.1. Слайди прикладу «Тема дня»

`split_slides` відкидає перший блок `--- … ---` (налаштування Marp) і повертає список `Slide(title, text, notes)`.

In [ ]:
slides = split_slides(EXAMPLE)
print(len(slides), "слайдів")
for number, slide in enumerate(slides, 1):
    print(f"{number:>2}. {slide.title:<28} слів у нотатках: {words(slide.notes):>3}")

### Приклад 5.2. Скільки слів бачить зал

Задача: знайти слайд, який зал **читатиме** замість слухати. `check()` рахує слова на слайді так: з посилань лишає підпис (адресу зал не читає), прибирає службові символи Markdown. Повторимо це окремою функцією, щоб побачити число для кожного слайда.

In [ ]:
LINK = re.compile(r"!?\[([^\]]*)\]\([^)]*\)")             # [підпис](адреса) → підпис
AUTOLINK = re.compile(r"<(?:https?|mailto):[^>]*>")


def visible_words(text: str) -> int:
    """Слова, які видно на слайді (так само рахує pitch_check.check)."""
    visible = AUTOLINK.sub(" ", LINK.sub(r"\1", text))
    return words(re.sub(r"[|`*>#-]", " ", visible))

In [ ]:
for slide in slides:
    n = visible_words(slide.text)
    print(f"{slide.title:<28} {n:>3} {'▇' * (n // 2)}{'  ← понад ліміт' if n > MAX_WORDS_ON_SLIDE else ''}")

Найбільше слів — на слайдах-таблицях (ринок, «що є · що далі»), але всі в межах 45. Решта тексту — в нотатках: її кажуть, а не показують.

### Приклад 5.3. `check()` на прикладі й на шаблоні

In [ ]:
print("приклад:", check(EXAMPLE) or "✓ готово до репетиції")
template_problems = check(TEMPLATE)
print(f"шаблон: {len(template_problems)} зауважень, перші три:")
for problem in template_problems[:3]:
    print("  ✗", problem)

Приклад проходить усе. Шаблон — ні, і так задумано: кожен слайд має `[заготовку]`, і `check()` не дасть забути хоч одну. Нотатки в шаблоні є (це підказки «що сказати»), тож зауваження — про заготовки на кожному слайді і про час: підказки — менше хвилини мови.

### Приклад 5.4. Те саме з командного рядка

Задача: перевіряти дек як тести — перед кожним «комітом» виступу. Скрипт повертає код виходу 0 («готово») або 1 («є зауваження»), тож його можна поставити в CI поруч з тестами.

In [ ]:
for name in ("example_news_digest_deck.md", "pitch_deck_template.md"):
    run = subprocess.run([sys.executable, str(HERE / "pitch_check.py"), str(HERE / name)], capture_output=True, text=True)
    lines = run.stdout.strip().splitlines()
    print(f"код виходу {run.returncode}: {lines[0]}")
    print("   ", lines[-1].strip())

### Задача 3. Бюджет часу на кожен слайд

Напиши дві функції:

- `seconds_per_slide(slides)` → список `(назва, секунд)`: скільки триває мова на кожному слайді за нотатками при темпі `WORDS_PER_MINUTE` (слова нотаток — `words(slide.notes)`; округли `round` до цілих);
- `longest_slide(timing)` → назва слайда з найдовшою мовою.

🔮 **Прогноз:** який слайд прикладу найдовший за мовою?

In [ ]:
def seconds_per_slide(slides) -> list[tuple[str, int]]:
    # BEGIN SOLUTION
    return [(s.title, round(words(s.notes) / WORDS_PER_MINUTE * 60)) for s in slides]
    # END SOLUTION


def longest_slide(timing: list[tuple[str, int]]) -> str:
    # BEGIN SOLUTION
    return max(timing, key=lambda item: item[1])[0]
    # END SOLUTION

In [ ]:
timing = seconds_per_slide(slides)
assert isinstance(timing, list) and len(timing) == len(slides), "поверни список: по одному (назва, секунд) на кожен слайд"
for title, seconds in timing:
    print(f"{title:<28} {seconds:>4} с  " + "▇" * (seconds // 5))
total = sum(seconds for _, seconds in timing)
print(f"разом мова: {total // 60} хв {total % 60} с; найдовший: {longest_slide(timing)}")
assert all(isinstance(s, int) for _, s in timing), "секунди — цілі числа (round)"
assert longest_slide(timing) == "Проблема" and 4 * 60 <= total <= 7 * 60
assert longest_slide([("a", 1), ("b", 9), ("c", 3)]) == "b"
print("✅ Задача 3 розв'язана")

<details>
<summary>Відповідь</summary>

«Проблема» — і це правильно: на ньому вирішується, чи слухатимуть далі. Демо коротке за словами, але довге за часом — там ти **показуєш**, а не говориш. Тому мова в прикладі — ~4 хвилини, а разом з демо — 5–7.
</details>

### Приклад 5.5. План часу з шаблону

Задача: шаблон підказує тривалість кожного слайда на початку нотатки («60 с. Історія однієї людини…»). Витягнемо ці числа регулярним виразом і складемо план виступу з часом початку кожного слайда — як схема «10 слайдів за 7 хвилин» у книзі.

In [ ]:
PLANNED = re.compile(r"^(\d+) с\.")


def timeline(markdown: str) -> list[tuple[str, str, int]]:
    """(початок мм:сс, назва, секунд за планом) для кожного слайда з позначкою «N с.» у нотатках."""
    rows, start = [], 0
    for slide in split_slides(markdown):
        match = PLANNED.match(slide.notes)
        if match:
            seconds = int(match.group(1))
            rows.append((f"{start // 60}:{start % 60:02d}", slide.title, seconds))
            start += seconds
    return rows

In [ ]:
plan_rows = timeline(TEMPLATE)
for start, title, seconds in plan_rows:
    print(f"{start:>5}  {title:<28} {seconds:>3} с")
planned_total = sum(seconds for *_, seconds in plan_rows)
print(f"разом за планом: {planned_total // 60} хв {planned_total % 60:02d} с")

Рівно 7 хвилин — верхня межа регламенту. Найдовші за планом: демо (90 с), проблема (60 с), рішення (50 с). Якщо на репетиції вийшло 8 хвилин, різати треба не демо, а слайди, де ти говориш замість показувати.

### Приклад 5.6. «Зміни приклад» з книги — запуском

Книга пропонує два експерименти: прибрати слайд «Демо» і перенести нотатки слайда «Проблема» на сам слайд. Задача: не вгадувати, що скаже скрипт, а запустити. Дві функції змінюють **текст** деку.

In [ ]:
SEPARATOR = re.compile(r"^---[ \t]*$", re.M)


def deck_parts(markdown: str) -> tuple[str, list[str]]:
    """(налаштування Marp, [тексти слайдів]) — розрізаємо за рядками ---."""
    _, front, *chunks = SEPARATOR.split(markdown)
    return front, chunks


def join_deck(front: str, chunks: list[str]) -> str:
    return "---" + front + "---" + "---".join(chunks)


def drop_slide(markdown: str, title: str) -> str:
    front, chunks = deck_parts(markdown)
    return join_deck(front, [c for c in chunks if f"# {title}\n" not in c])


def notes_to_slide(markdown: str, title: str) -> str:
    front, chunks = deck_parts(markdown)
    moved = [re.sub(r"<!--(.*?)-->", r"\1", c, flags=re.S) if f"# {title}\n" in c else c for c in chunks]
    return join_deck(front, moved)

In [ ]:
assert join_deck(*deck_parts(EXAMPLE)) == EXAMPLE              # розрізали й склеїли — нічого не загубили
print("без «Демо»:")
for problem in check(drop_slide(EXAMPLE, "Демо")):
    print("  ✗", problem)
print("нотатки «Проблеми» на слайді:")
for problem in check(notes_to_slide(EXAMPLE, "Проблема")):
    print("  ✗", problem)

Збігається з книгою. Без демо — два зауваження: немає обов'язкового розділу, і разом з демо зникла розповідь під час нього (3,7 хв < 4). Слайдів 9 — ще в межах 8–12. Нотатки на слайді — три: 154 слова на слайді, слайд без нотаток, і виступ «коротшає» до 3,1 хв: текст на слайді скрипт не вважає мовою. Слайд перетворився на абзац, який зал читає замість слухати.

### Приклад 5.7. Свій дек із шаблону — кодом

Задача: почати свій дек не з порожнього файлу. Невелика зміна — підставимо в шаблон кілька заготовок зі словника, запишемо `my_deck.md` і подивимось, скільки `[дужок]` лишилось. Так видно прогрес: від 10 зауважень до «готово до репетиції».

In [ ]:
FILL = {
    "[Назва продукту]": "Тема дня",
    "[Одне речення: що змінюється в житті користувача]": "Новини твоєї галузі — у Telegram за 5 хвилин",
    "[Ім'я] · фінальний проєкт курсу PY": "Студентка курсу PY · фінальний проєкт",
    "Замість усіх [дужок] — свій текст; коментарі — те, що ти скажеш уголос.":
        "Я — студентка курсу PY. «Тема дня» надсилає новини твоєї галузі в Telegram замість години ранкового читання.",
    "[одна конкретна людина і її ситуація]": "PR-менеджерка стежить за 8–10 сайтами щоранку",
    "[скільки часу / грошей / нервів це коштує]": "~1 год щоранку, і все одно пропускає згадки",
    "[чому нинішні способи не підходять]": "агрегатори показують усе, а не її теми",
}


PLACEHOLDERS = re.compile(r"\[[^\]]+\](?!\()")          # [заготовка], але не [підпис](посилання)


def fill_template(template: str, values: dict[str, str]) -> str:
    for placeholder, text in values.items():
        template = template.replace(placeholder, text)
    return template

In [ ]:
my_deck = WORK / "my_deck.md"
my_deck.write_text(fill_template(TEMPLATE, FILL), encoding="utf-8")
mine = my_deck.read_text(encoding="utf-8")
before, after = PLACEHOLDERS.findall(TEMPLATE), PLACEHOLDERS.findall(mine)
print(f"записано {my_deck.name}: {my_deck.stat().st_size} байт; заготовок було {len(before)}, лишилось {len(after)}")
print("зауважень check():", len(check(TEMPLATE)), "→", len(check(mine)))
print("перше зауваження тепер:", check(mine)[0])

Перші два слайди вже чисті (ще й нотатка першого — справжня мова, а не підказка); решта чекає на тебе. Далі в терміналі: `python pitch_check.py my_deck.md` — доки не з'явиться `✓ готово до репетиції`.

---
## 6. Знайди помилку

### Приклад 6.1. Чернетка, яка «виглядає нормально»

Перша чернетка деку. Задача: знайти, які слайди переробити, — не очима, а за повідомленнями `check()` виду `слайд N «…»`.

In [ ]:
DRAFT = """---
marp: true
---
# Мій фінальний проєкт

<!-- Привіт, я розкажу про свій проєкт. -->
---
# Технології

Python 3.13, FastAPI, SQLAlchemy 2, Alembic, PostgreSQL 16, Redis 7, aiogram 3, Docker, Docker Compose,
nginx, GitHub Actions, pytest, mypy, httpx, Pydantic 2, Gemini API, Anthropic API, bcrypt, PyJWT, Marp,
BeautifulSoup, aiohttp, uvicorn, fakeredis, factory, Playwright, Channels, daphne, Django 5.2, DRF,
SQLite, Jupyter, Colab, pandas, Dash, Streamlit, gRPC, Postman, Linux, bash.

<!-- Ось усі технології, які я використав. -->
---
# Ринок

Ринок — мільярди доларів, візьмемо [X]%.

<!-- TODO -->
---
# Дякую за увагу!

<!-- Питання? -->
"""

In [ ]:
draft_problems = check(DRAFT)
for problem in draft_problems:
    print("✗", problem)
fixes = {int(m.group(1)) for p in draft_problems if (m := re.match(r"слайд (\d+)", p))}
print("переробити слайди:", sorted(fixes))
assert fixes == {2, 3}
assert "немає слайда «проблема»" in draft_problems and "слайдів 4, треба 8–12" in draft_problems

<details>
<summary>Що не так, крім того, що бачить скрипт</summary>

1. **Немає проблеми** — дек починається з «мій проєкт», а не з людини і її болю.
2. **Слайд технологій** — список для резюме, а не для питчу. Техліду цікаве **одне** рішення з виміром, інвестору й клієнту — жодне.
3. **Ринок згори вниз** (приклад 4.1) і заготовка `[X]`.
4. **«Дякую за увагу»** замість прохання: слухач не знає, що від нього хочуть.

Скрипт ловить форму (кількість, обсяг, час, заготовки, розділи). Чи переконливий зміст — вирішують люди. Тому після `✓ готово до репетиції` — дві репетиції вголос.
</details>

### Приклад 6.2. Слайд «Ринок» абзацом

Типова перша чернетка слайда з книги. 🔮 **Прогноз:** скільки слів на ньому і чи зловить його `check()` як «задовгий»?

In [ ]:
MARKET_SLIDE = """# Ринок

Світовий ринок новинних агрегаторів оцінюється в мільярди доларів
і щороку зростає. Якщо ми займемо лише 1% ринку, то наш дохід
складе десятки мільйонів доларів на рік, що дозволить масштабуватися.
"""

In [ ]:
body = MARKET_SLIDE.split("\n", 1)[1]
print("слів в абзаці:", words(body), "| слів на слайді з заголовком:", visible_words(MARKET_SLIDE), "| ліміт:", MAX_WORDS_ON_SLIDE)

<details>
<summary>Відповідь</summary>

29 слів — менше за ліміт 45, тож за обсягом скрипт його **пропустить**. Помилки тут змістові: згори вниз, «мільярди» без джерела, світовий ринок для продукту, що читає українські сайти, і суцільний абзац, який зал читатиме. Краще — таблиця TAM → SAM → SOM з позначеними припущеннями і формула доходу (задача 2).
</details>

---
## 7. Оцінювання і репетиції

### Приклад 7.1. Критерії як дані

Задача: оцінити себе перед виступом тими самими критеріями, що й викладач. Таблиця з книги — словник `{критерій: максимум}`; функція перевіряє, що бали в межах, і показує найслабше місце.

In [ ]:
RUBRIC = {"проблема": 20, "рішення і демо": 25, "технічна глибина": 25, "чесність і план": 10, "подача": 20}


def grade(scores: dict[str, int]) -> str:
    """Сума балів і критерій з найбільшою втратою; бали поза межами — помилка."""
    for name, value in scores.items():
        if not 0 <= value <= RUBRIC[name]:
            raise ValueError(f"{name}: {value} поза межами 0–{RUBRIC[name]}")
    lost = {name: RUBRIC[name] - scores.get(name, 0) for name in RUBRIC}
    weakest = max(lost, key=lost.get)
    return f"{sum(scores.values())}/{sum(RUBRIC.values())}; найбільше втрачено: {weakest} (−{lost[weakest]})"

In [ ]:
self_check = {"проблема": 12, "рішення і демо": 22, "технічна глибина": 20, "чесність і план": 8, "подача": 15}
print(grade(self_check))
try:
    grade({**self_check, "чесність і план": 15})
except ValueError as error:
    print("помилка:", error)

Найбільше втрачено на «проблемі»: немає жодного факту з розмови. Це і є план на тиждень до уроку 53 — два інтерв'ю за The Mom Test, а не ще один слайд.

### Приклад 7.2. Дві репетиції з секундоміром

Задача: перша репетиція завжди довша, ніж здається. Записуємо секунди на кожен слайд і порівнюємо з планом шаблону (приклад 5.5): де перевищення понад 20%.

In [ ]:
REHEARSALS = {
    "репетиція 1": [25, 95, 60, 120, 30, 55, 35, 45, 50, 20],
    "репетиція 2": [20, 65, 50, 95, 30, 40, 30, 40, 35, 15],
}


def overruns(measured: list[int], plan_rows: list[tuple[str, str, int]], tolerance: float = 0.2) -> list[str]:
    """Слайди, де виміряний час перевищує план більше ніж на tolerance."""
    return [f"{title} {sec} с замість {planned}" for sec, (_, title, planned) in zip(measured, plan_rows)
            if sec > planned * (1 + tolerance)]

In [ ]:
for name, measured in REHEARSALS.items():
    total = sum(measured)
    verdict = "у регламенті" if 5 * 60 <= total <= 7 * 60 else "поза регламентом 5–7 хв"
    print(f"{name}: {total // 60}:{total % 60:02d} ({verdict}); перевищення: {overruns(measured, plan_rows) or 'немає'}")

Перша репетиція — 8:55, і майже весь зайвий час — на проблемі, демо й ринку. Друга — рівно 7:00: перевищень понад 20% немає, але й запасу немає — наступна мета 6 хвилин. Тепер відомо, **де** різати.

### Приклад 7.3. PDF через Marp — якщо є Node.js

Marp збирає PDF чи PowerPoint з того самого Markdown. Йому потрібні Node.js (`npx`) і Chrome/Edge/Chromium, а `npx` завантажує пакет з інтернету, тож у ноутбуці ми його не запускаємо — лише перевіряємо, чи є інструменти, і друкуємо команди.

In [ ]:
for tool in ("npx", "chromium", "google-chrome", "msedge"):
    print(f"{tool:<14} {'є: ' + shutil.which(tool) if shutil.which(tool) else 'немає'}")
print("\nкоманди (з папки уроку):")
print("  npx @marp-team/marp-cli my_deck.md --pdf --pdf-notes   # PDF з нотатками")
print("  npx @marp-team/marp-cli my_deck.md --pptx              # PowerPoint")
print("готовий приклад:", (HERE / "example_news_digest_deck.pdf").name, f"{(HERE / 'example_news_digest_deck.pdf').stat().st_size // 1024} КБ")

Зручніше Google Slides, PowerPoint чи Canva — теж добре: структура і правила ті самі, а `check()` тоді заміниш чек-листом з книги.

### Прибираємо

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("тимчасова папка прибрана:", not WORK.exists())

---
## Самоперевірка

1. Чому «неуважна» — не корінь, хоч і листок дерева?
2. Чому «ліміт 3 замість 5» — виправлення симптому, а не кореня?
3. Яке з питань погане і чому: «Як ви зараз стежите за новинами?» чи «Чи зручно було б отримувати новини в Telegram?»
4. Чому дохід рахують від SOM, а не від TAM?
5. Чому слайд «Проблема» найдовший за мовою, а «Демо» — ні?
6. Що `check()` не перевіряє взагалі?

<details>
<summary>Відповіді</summary>

1. Це ярлик: продуктом його не змінити. Корінь — те, на що можна вплинути.
2. З N репліками і кешем у пам'яті спроб усе одно 3·N; корінь — неспільний кеш, і лише спільний кеш дає рівно ліміт.
3. Друге: майбутнє («було б») і твоє рішення (Telegram). Перше питає про теперішню практику.
4. SOM — ті, кого реально досягнеш; TAM — стеля, а не дохід.
5. На демо ти показуєш, а не говориш: час іде на дії, а не на слова.
6. Зміст: чи справжня проблема, чи переконливе демо, чи чесні числа. Це вирішують люди — на репетиціях і на уроці.
</details>

## Далі

- Книга курсу: [Урок 53](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_53/) — кому продаєш, 5 «чому» з Toyota та його межі, The Mom Test, структури деків, демо, питання після виступу, критерії оцінювання.
- У папці уроку: `pitch_deck_template.md` — заповни, перевір `python pitch_check.py мій_дек.md`, зроби PDF: `npx @marp-team/marp-cli мій_дек.md --pdf`.
- [Бонус. CV розробника](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/bonus_cv/) — ті самі факти про проєкт, що й у питчі, — у CV.